In [1]:
import pandas as pd
import sys
from pathlib import Path
import os 
from dotenv import load_dotenv

In [2]:
load_dotenv(r"D:\CHATBOT\.env")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")


In [3]:
#print(GEMINI_API_KEY)

In [4]:
df = pd.read_csv(r"D:\CHATBOT\BotConvMem\app\graph\product_documents.csv")
df.head()

,article_id,text
0,108775015,\nProduct Name: Strap top\n\nProduct Type: Ves...
1,108775044,\nProduct Name: Strap top\n\nProduct Type: Ves...
2,108775051,\nProduct Name: Strap top (1)\n\nProduct Type:...
3,110065001,\nProduct Name: OP T-shirt (Idro)\n\nProduct T...
4,110065002,\nProduct Name: OP T-shirt (Idro)\n\nProduct T...


In [5]:
row = df.iloc[0]

print("Article ID:", row["article_id"])
print("Text:", row["text"][:200])

Article ID: 108775015
Text: 
Product Name: Strap top

Product Type: Vest top

Product Group: Garment Upper body

Color: Black

Department: Jersey Basic

Index: Ladieswear

Index Group: Ladieswear

Section: Womens Everyday Basics


In [6]:
print(df.shape)
print(df["article_id"].dtype)
print(df[["article_id", "text"]].head(2))

(105542, 2)
int64
   article_id                                               text
0   108775015  \nProduct Name: Strap top\n\nProduct Type: Ves...
1   108775044  \nProduct Name: Strap top\n\nProduct Type: Ves...


In [7]:
APP_DIR = Path.cwd().parent
sys.path.insert(0, str(APP_DIR))

from database.db import get_connection

In [8]:
conn = get_connection()
cursor = conn.cursor()

Connecting to connaq-dev-server.database.windows.net (using mode: SQL...
pyodbc version: 5.3.0
Database connection failed.
Error details: ('HY000', "[HY000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Database 'connaq-dev-db' on server 'connaq-dev-server.database.windows.net' is not currently available.  Please retry the connection later.  If the problem persists, contact customer support, and provide them the session tracing ID of '{6F590074-D9EA-439A-83F8-C7DA6B7559DD}'. (40613) (SQLDriverConnect); [HY000] [Microsoft][ODBC Driver 18 for SQL Server]Invalid connection string attribute (0); [HY000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Database 'connaq-dev-db' on server 'connaq-dev-server.database.windows.net' is not currently available.  Please retry the connection later.  If the problem persists, contact customer support, and provide them the session tracing ID of '{6F590074-D9EA-439A-83F8-C7DA6B7559DD}'. (40613); [HY000] [Microsoft][ODBC Driver 18 for SQL Se

Error: ('HY000', "[HY000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Database 'connaq-dev-db' on server 'connaq-dev-server.database.windows.net' is not currently available.  Please retry the connection later.  If the problem persists, contact customer support, and provide them the session tracing ID of '{6F590074-D9EA-439A-83F8-C7DA6B7559DD}'. (40613) (SQLDriverConnect); [HY000] [Microsoft][ODBC Driver 18 for SQL Server]Invalid connection string attribute (0); [HY000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Database 'connaq-dev-db' on server 'connaq-dev-server.database.windows.net' is not currently available.  Please retry the connection later.  If the problem persists, contact customer support, and provide them the session tracing ID of '{6F590074-D9EA-439A-83F8-C7DA6B7559DD}'. (40613); [HY000] [Microsoft][ODBC Driver 18 for SQL Server]Invalid connection string attribute (0)")

In [ ]:
'''
for _, row in df.iterrows():
    text = row["text"]
    print(text)
'''
    # generate embedding
    # update Neo4j

'\nfor _, row in df.iterrows():\n    text = row["text"]\n    print(text)\n'

In [ ]:
print(df.columns.tolist())

['article_id', 'text']


In [ ]:
update_query = """
UPDATE Products
SET product_text = ?
WHERE article_id = ?
"""
data = [
    (
        row["text"],
        int(row["article_id"])
    )
    for _, row in df.iterrows()
]

print("Rows affected:", cursor.rowcount)

conn.commit()

Rows affected: -1


In [ ]:
print(len(data))
print(data[0][1])
print(data[0][0][:100])

105542
108775015

Product Name: Strap top

Product Type: Vest top

Product Group: Garment Upper body

Color: Black

D


In [ ]:
test_data = data[:1]

cursor.executemany(update_query, test_data)
conn.commit()

print("One row updated")

One row updated


In [ ]:
cursor.execute("""
    SELECT article_id
    FROM Products
    WHERE product_text IS NULL
""")

null_ids = {row[0] for row in cursor.fetchall()}

print("Remaining:", len(null_ids))

Remaining: 17542


In [ ]:
data = [
    (
        row["text"],
        int(row["article_id"])
    )
    for _, row in df.iterrows()
    if int(row["article_id"]) in null_ids
]

In [ ]:
update_query = """
UPDATE Products
SET product_text = ?
WHERE article_id = ?
  AND product_text IS NULL
"""

BATCH_SIZE = 1000

for start in range(0, len(data), BATCH_SIZE):

    batch = data[start:start + BATCH_SIZE]

    cursor.executemany(update_query, batch)
    conn.commit()

    completed = min(start + BATCH_SIZE, len(data))

    print(f"Updated {completed:,} / {len(data):,}")

Updated 1,000 / 17,542
Updated 2,000 / 17,542
Updated 3,000 / 17,542
Updated 4,000 / 17,542
Updated 5,000 / 17,542
Updated 6,000 / 17,542
Updated 7,000 / 17,542
Updated 8,000 / 17,542
Updated 9,000 / 17,542
Updated 10,000 / 17,542
Updated 11,000 / 17,542
Updated 12,000 / 17,542
Updated 13,000 / 17,542
Updated 14,000 / 17,542
Updated 15,000 / 17,542
Updated 16,000 / 17,542
Updated 17,000 / 17,542
Updated 17,542 / 17,542


In [ ]:
client = genai.Client(GEMINI_API_KEY)

NameError: name 'genai' is not defined

In [ ]:
response = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Product Name: Strap Top. Color: Black. Department: Ladieswear."
)

embedding = response.embeddings[0].values

print(len(embedding))
print(embedding[:10])

In [ ]:
row = df.iloc[0]

print("Article ID:", row["article_id"])
print("Text:", row["text"][:200])

In [ ]:
'''
UNWIND $rows AS row

MATCH (p:Product {
    article_id: row.article_id
})

SET p.embedding = row.embedding
'''

In [ ]:
'''
CREATE VECTOR INDEX product_embedding_index
FOR (p:Product)
ON p.embedding
OPTIONS {
    indexConfig: {
        `vector.dimensions`: 3072,
        `vector.similarity_function`: 'cosine'
    }
}
'''